***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [7. 观测系统](7_0_introduction.ipynb)
    * 上一节： [7.3 模拟电子学](7_3_analogue.ipynb)
    * 下一节： [7.5 主波束](7_5_primary_beam.ipynb)

***


## 7.4 数字相关器：采样、通道化与相关

上一节把信号从射频搬到适合处理的中频，并讨论了模拟增益、带通和噪声。数字相关器接过的是每台天线、每个极化通道的一串电压样本。它要完成的工作可以概括为三步：先把连续电压采样并量化为数字流，再把宽带时域样本通道化为许多窄带频率通道，最后在每个时间、频率和极化通道上对所有天线对做互相关。

FX 相关器的核心形式可以写成

$$
V_{pq}(\nu_k)=\left\langle X_p(\nu_k,t)X_q^*(\nu_k,t)\right\rangle_t,
$$

其中 $X_p(\nu_k,t)$ 是第 $p$ 台天线在第 $k$ 个频率通道中的复电压，星号表示复共轭，角括号表示在若干时间块上积分平均。这个式子看起来很接近 RIME 中的可见度，但它强调的是工程实现：真实可见度不是一次乘法得到的，而是海量采样、通道化、延迟补偿、乘累加和数据组织共同产生的数据产品。


### 7.4.1 ADC、奈奎斯特采样与量化

数字系统的第一步是 ADC。若采用实数采样，且输入信号最高频率为 $\nu_{\max}$，采样率必须满足奈奎斯特条件

$$
\nu_s>2\nu_{\max}.
$$

若不满足，频率高于 $\nu_s/2$ 的分量会折叠回低频，形成混叠。对一个真实频率 $\nu$，其在采样后落到的表观频率可以理解为距离最近采样谐波的余量，常写成

$$
\nu_{\rm alias}=\left|\nu-n\nu_s\right|,
$$

其中整数 $n$ 选到使结果落在 $[0,\nu_s/2]$ 内。混叠一旦发生，后续数字处理无法知道该低频分量究竟来自真实低频还是折叠高频，因此 ADC 之前必须有模拟抗混叠滤波。

采样后的电压还要量化。$n$ bit ADC 给出 $2^n$ 个幅度等级；比特数越低，数据率越低，但量化噪声和相关损失越大。对干涉仪而言，量化误差不只影响单天线功率谱，也会改变相关系数的统计关系，因此低比特相关器需要量化效率校正或 Van Vleck 类修正。


![采样率不足时的混叠](figures/adc_aliasing_sampling.png)

**图 7.4.1** 同一个中频信号在不同采样率下的表现。64 MHz 采样可以恢复 8 MHz 和 18 MHz 两个分量；30 MHz 采样的奈奎斯特频率只有 15 MHz，因此 18 MHz 分量会折叠到 12 MHz 附近。


![量化比特数与波形保真度](figures/adc_quantization_depth.png)

**图 7.4.2** 同一段归一化电压在不同量化位深下的表示。2 bit 量化明显呈台阶状，4 bit 已显著改善，8 bit 更接近连续波形。低比特量化在工程上节省数据率，但会改变相关统计。


### 7.4.2 通道化：FFT、窗函数与 PFB 的动机

ADC 输出的是时域样本，而 RIME 与成像通常按频率通道组织可见度。因此，相关器必须先进行通道化。最直接的通道化方法，是把长度为 $N$ 的时间块做离散 Fourier 变换：

$$
X_k=\sum_{n=0}^{N-1}x_n\exp\left(-2\pi i\frac{kn}{N}\right),
$$

通道间隔为

$$
\Delta\nu=\frac{\nu_s}{N}.
$$

如果信号频率恰好落在 FFT 栅格上，能量主要集中在一个通道；若不在栅格上，有限时间窗等价于在频域与窗函数谱卷积，能量会泄漏到旁边通道。这种谱泄漏在强源、强 RFI 或高动态范围谱线观测中尤其麻烦。

窗函数可以降低旁瓣，但会加宽主瓣并改变通道响应。多相滤波器组（polyphase filter bank, PFB）进一步把“加窗”和“FFT”组合成多抽头滤波结构，使每个输出通道的频响更接近理想带通，并减小通道间串扰。现代宽带阵列通常会认真设计 F-engine，原因就在这里：通道化质量会直接进入带通校准、RFI 标记和谱线保真度。


![窗函数、谱泄漏与通道化](figures/fft_window_channelization.png)

**图 7.4.3** 左图比较矩形窗和 Hann 窗对非栅格频率信号的谱泄漏；右图展示宽带信号经块 FFT 后的通道化功率谱。PFB 的目标，是在可接受主瓣宽度下进一步压低旁瓣和通道串扰。


### 7.4.3 延迟补偿、相位旋转与 FX 可见度

对第 $p$、$q$ 两台天线，几何延迟 $\tau_g$ 会让同一个波前到达两台天线的时间不同。在频域中，时间延迟表现为线性相位斜率：

$$
X_q(\nu)=X_p(\nu)\exp(-2\pi i\nu\tau_g).
$$

若直接相关，得到的可见度相位随频率快速变化。对宽带积分来说，不同频率通道的相位会彼此抵消，造成带宽去相干；对长时间积分来说，地球自转使几何延迟随时间变化，还会造成时间平均去相干。因此相关器必须使用延迟模型。

实际系统通常把延迟补偿分成两层。整数采样级的粗延迟可在时域移动样本；分数采样级的精细延迟通常在频域乘上线性相位因子实现。除此之外，针对选定相位中心的条纹旋转（fringe rotation）会持续更新相位，使目标方向在积分过程中保持相干。校准和成像中看到的相位，已经是这些实时模型处理之后的结果。


![延迟相位斜率与补偿](figures/delay_correction_visibility.png)

**图 7.4.4** 几何延迟会在频域中产生可见度相位斜率。乘以相反的线性相位因子后，斜率被去除，宽带频率通道可以相干平均。真实阵列中，延迟模型随时间、基线和相位中心持续变化。


### 7.4.4 XF 等价性与重采样代价

FX 相关器先做 Fourier 变换，再在频域中做互相关。另一类历史上常见的实现是 XF 或 lag correlator：先计算不同时间滞后的相关函数

$$
R_{pq}(\tau)=\left\langle x_p(t)x_q^*(t-\tau)\right\rangle,
$$

再对 $R_{pq}(\tau)$ 做 Fourier 变换得到谱域可见度。理想条件下，这两条路径通过 Wiener-Khinchin 关系等价。差别主要来自实现细节：有限窗、抽样时滞数量、量化、积分方式和滤波器响应都会让实际输出产生小差异。

相关器内部还常常进行重采样或重新量化。F-engine 输出若保留过高位宽，corner turn 和 X-engine 的数据搬运压力会很大；若过早降位宽，则会损失相关效率。1 bit 量化是最极端的例子。对零均值高斯变量，1 bit 符号相关 $c$ 与真实相关系数 $\rho$ 满足近似关系

$$
c=\frac{2}{\pi}\arcsin\rho,
\qquad
\rho=\sin\left(\frac{\pi c}{2}\right).
$$

这就是 Van Vleck 修正的基本思想之一：低比特相关器不是不能用，但必须理解量化改变了统计量本身。


![量化对相关系数的影响](figures/quantization_correlation_recovery.png)

**图 7.4.5** 低比特量化会压低原始相关估计。1 bit 符号相关经过 Van Vleck 型反变换后可以部分恢复真实相关系数；2 bit 量化损失较小，但仍需要校正或标定。


### 7.4.5 Corner Turn、相关矩阵与算量扩展

F-engine 的自然数据布局通常按天线和时间组织：每台天线的电压流被通道化为频谱块。X-engine 的自然需求却相反：对某一个频率通道，需要同时拿到所有天线、所有极化的电压，才能形成全阵列相关矩阵。把数据从“按天线组织”转成“按频率组织”的全局转置，称为 corner turn。

这个步骤不改变物理信息，却常常决定系统是否可扩展。对 $N_{\rm ant}$ 台天线，互相关基线数为

$$
N_{\rm bl}=\frac{N_{\rm ant}(N_{\rm ant}-1)}{2},
$$

若包括自相关，则为 $N_{\rm ant}(N_{\rm ant}+1)/2$。X-engine 的乘累加规模随 $N_{\rm ant}^2$ 增长，而 F-engine 的 FFT 成本大致随 $N_{\rm ant}N_{\rm chan}\log N_{\rm chan}$ 增长。当天线数变大时，数据交换和 X-engine 乘累加很快成为主要瓶颈。


![相关矩阵与基线数扩展](figures/corner_turn_scaling.png)

**图 7.4.6** 左图示意 7 台天线的相关矩阵，对角线为自相关，非对角线为互相关。右图显示互相关基线数随天线数按二次关系增长。相关器设计中的难点往往不是单台天线 FFT，而是如何搬运并相关所有天线的同频通道。


### 7.4.6 数据产品：Measurement Set 的最小骨架

相关器的输出不能只是一个复数数组。后续校准和成像必须知道每条可见度对应哪两台天线、哪个时间、哪个频率通道、哪个极化积、什么 $uvw$ 坐标、什么权重，以及哪些数据已经被标记为无效。业界常用的数据容器是 Measurement Set（MS）。完整 MS 标准很丰富，但它的核心直觉很简单：MAIN 表存放逐行可见度，子表存放解释这些可见度所需的元数据。

一行 MAIN 表通常对应一个时间上的一条基线，`DATA` 列则包含该行所有频率通道和极化积的复可见度。`UVW` 给出该基线在当前时间和相位中心下的坐标，`WEIGHT` 或 `SIGMA` 给出噪声权重，`FLAG` 表示哪些通道或极化积不可用。ANTENNA、SPECTRAL_WINDOW、POLARIZATION、FIELD 等子表则分别说明天线位置、频率布局、极化相关积和相位中心。


![Measurement Set 的最小骨架](figures/measurement_set_minimal_skeleton.png)

**图 7.4.7** 可见度行只有连同元数据一起才有物理意义。MAIN 表给出基线、时间、$uvw$、复数据、权重和标记；子表描述天线、频率、极化、场中心和观测状态。校准与成像软件正是依靠这些表把复数数组解释为射电观测。


本节把数字后端的主线串成了一条链：ADC 采样和量化把模拟电压变成数据流；通道化把宽带时域信号变成窄带频域电压；延迟补偿和条纹旋转保证目标方向相干；X-engine 对所有天线对形成相关矩阵；最终数据连同时间、频率、极化、$uvw$、权重和标记一起写入观测数据产品。后续几节讨论的主波束、极化、传播效应和 RFI，都会作用在这些数字可见度及其元数据上。

***

* 下一节： [7.5 主波束](7_5_primary_beam.ipynb)
